# Live conference demo launcher

Run the next cell once before your talk. It starts `app.py` (the real Streamlit workbench) in the background and waits until its port is actually accepting connections, then prints the URL to open. Because this is a VS Code remote/SSH session, VS Code should also auto-detect the port and pop up an "Open in Browser" notification (or check the **Ports** tab in the bottom panel) so you can share it with the audience.

Default mode inside the app is **"Browse a real DebateGPT example"**, which needs no GPU and no HF_TOKEN and loads instantly -- the reliable path for the live session. Switch to **"Generate a new debate live"** in the app's sidebar only if you want the GPU-backed generation for the wow factor.

Run the last cell after your talk to stop the server.

In [1]:
import socket
import subprocess
import sys
import time
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent

PORT = 8501


def _port_open(port: int, host: str = '127.0.0.1') -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0


if _port_open(PORT):
    print({'status': 'already_running', 'url': f'http://localhost:{PORT}', 'note': 'stop it with the last cell first if you want a clean restart'})
else:
    log_path = root / 'logs' / 'streamlit_demo.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_handle = open(log_path, 'w', encoding='utf-8')
    streamlit_proc = subprocess.Popen(
        [
            sys.executable, '-m', 'streamlit', 'run', str(root / 'app.py'),
            '--server.port', str(PORT),
            '--server.headless', 'true',
            '--server.address', '0.0.0.0',
            '--browser.gatherUsageStats', 'false',
        ],
        cwd=str(root), stdout=log_handle, stderr=subprocess.STDOUT,
    )

    ready = False
    for _ in range(60):
        if _port_open(PORT):
            ready = True
            break
        time.sleep(0.5)

    print({
        'status': 'ready' if ready else 'still_starting_check_log',
        'pid': streamlit_proc.pid,
        'url': f'http://localhost:{PORT}',
        'log': str(log_path),
        'note': (
            'Look for an "Open in Browser" notification from VS Code, or open the '
            'Ports panel (bottom bar) and click the globe icon next to 8501.'
        ),
    })

{'status': 'already_running', 'url': 'http://localhost:8501', 'note': 'stop it with the last cell first if you want a clean restart'}


In [3]:
# Run this after your talk to free the port.
import subprocess

PORT = 8501
result = subprocess.run(['pkill', '-f', f'streamlit run.*--server.port {PORT}'], capture_output=True, text=True)
print({'status': 'stopped' if result.returncode == 0 else 'nothing_was_running'})

{'status': 'stopped'}
